In [1]:
# Preprocessing Pipeline
# Goal: Clean data, encode variables, split into train/test
# preserving subgroup labels for fairness auditing

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Libraries loaded")

Libraries loaded


In [2]:
# Load the raw data
df = pd.read_csv("../data/raw/heart_failure.csv")

# Keep a copy of sensitive attributes for fairness auditing later
sensitive_cols = ['sex', 'age']

print("Data loaded:", df.shape)
df.head()

Data loaded: (299, 13)


,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


In [3]:
# Create age group column for fairness auditing
df['age_group'] = pd.cut(df['age'], 
                          bins=[0, 50, 60, 70, 100], 
                          labels=['<50', '50-60', '60-70', '70+'])

# Separate features, target, and sensitive attributes
X = df.drop(columns=['DEATH_EVENT', 'age_group'])
y = df['DEATH_EVENT']
sensitive = df[['sex', 'age_group']]

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeature columns:", X.columns.tolist())

Features shape: (299, 12)
Target shape: (299,)

Feature columns: ['age', 'anaemia', 'creatinine_phosphokinase', 'diabetes', 'ejection_fraction', 'high_blood_pressure', 'platelets', 'serum_creatinine', 'serum_sodium', 'sex', 'smoking', 'time']


In [4]:
# Scale the numerical features
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print("Before scaling - age mean:", round(X['age'].mean(), 2))
print("After scaling - age mean:", round(X_scaled['age'].mean(), 2))
print("\nScaling complete")

Before scaling - age mean: 60.83
After scaling - age mean: 0.0

Scaling complete


In [5]:
# Split into train and test sets
# stratify=y ensures equal proportion of deaths in both sets
X_train, X_test, y_train, y_test, sensitive_train, sensitive_test = train_test_split(
    X_scaled, y, sensitive, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("\nDeath rate in train:", round(y_train.mean(), 3))
print("Death rate in test:", round(y_test.mean(), 3))

Train size: (239, 12)
Test size: (60, 12)

Death rate in train: 0.322
Death rate in test: 0.317


In [6]:
# Save processed data for use in future notebooks
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)
sensitive_train.to_csv("../data/processed/sensitive_train.csv", index=False)
sensitive_test.to_csv("../data/processed/sensitive_test.csv", index=False)

print("All processed files saved successfully")

All processed files saved successfully
